# 03 — Genetic Algorithm (GA) Feature Selection

Binary GA searches for a compact feature mask. This notebook uses the shared experiment pipeline so all optimizers receive the same data splits, classifiers, seeds, and evaluation rules.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "utils").exists():
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT))

import random
import numpy as np


## Binary Genetic Algorithm

In [ ]:
# ----------------------------
# Binary Genetic Algorithm
# ----------------------------

def initialize_population(pop_size, n_features):
    return np.random.randint(0, 2, (pop_size, n_features))


def tournament_selection(population, fitness, k=3):
    idx = np.random.choice(len(population), k, replace=False)

    tournament_fitness = fitness[idx]

    winner = idx[np.argmin(tournament_fitness)]

    return population[winner].copy()


def crossover(parent1, parent2, pc=0.9):
    if random.random() > pc:
        return parent1.copy(), parent2.copy()

    point = random.randint(1, len(parent1)-2)

    child1 = np.concatenate((parent1[:point], parent2[point:]))
    child2 = np.concatenate((parent2[:point], parent1[point:]))

    return child1, child2


def mutation(individual, pm=0.05):
    child = individual.copy()

    for i in range(len(child)):
        if random.random() < pm:
            child[i] = 1 - child[i]

    if np.sum(child) == 0:
        child[random.randint(0, len(child)-1)] = 1

    return child


def run_ga(obj_func,
           n_features,
           pop_size=30,
           generations=50,
           pc=0.9,
           pm=0.05):

    population = initialize_population(pop_size, n_features)

    history = []

    best_solution = None
    best_fitness = np.inf

    for g in range(generations):

        fitness = np.array([obj_func(ind) for ind in population])

        idx = np.argmin(fitness)

        if fitness[idx] < best_fitness:
            best_fitness = fitness[idx]
            best_solution = population[idx].copy()

        history.append(best_fitness)

        # Do not create a final population that will never be evaluated.
        if g == generations - 1:
            break

        new_population = []

        while len(new_population) < pop_size:

            p1 = tournament_selection(population, fitness)
            p2 = tournament_selection(population, fitness)

            c1, c2 = crossover(p1, p2, pc)

            c1 = mutation(c1, pm)
            c2 = mutation(c2, pm)

            new_population.extend([c1, c2])

        population = np.array(new_population[:pop_size])

    return best_solution, best_fitness, history

## Run the complete feature-selection experiment

The shared experiment runner supplies the configured datasets, classifiers,
optimizer seeds, population size, and iteration count. Feature selection uses
the validation set. The test set is evaluated only after the final mask has
been selected.


In [ ]:
from utils.experiments import run_feature_selector


def ga_runner(
    objective,
    n_features,
    pop_size,
    iterations,
):
    return run_ga(
        obj_func=objective,
        n_features=n_features,
        pop_size=pop_size,
        generations=iterations,
        pc=0.9,
        pm=0.05,
    )


ga_results = run_feature_selector("GA", ga_runner)
ga_results.tail()
